# Task 09 -- Growing DMD modes (`docs/open_questions.md` §5.5)

Surveys the distribution of `|lambda_k|` across ranks/windows to
distinguish genuine growth from noise-attributable growth, per
`dynamic_mode_decomposition.md` §6's proposed clip-and-flag mitigation.
See `tasks/09-growing-dmd-modes.md`.

In [ ]:
from pathlib import Path

import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)
GAUSS_DIR = Path("../results/task07/gaussianize")

# Redefined 2026-09-29 (task 07, tasks/09-growing-dmd-modes.md "Update"):
# reuses the production 40% Tucker core's Gaussianized core directly,
# instead of fitting a brand-new small Tucker core on a synthetic era-3
# window -- no Tucker fit needed here at all, this is already the real,
# production-scale core (era 2, 9-month window, rank (248,576)).
core = np.load(GAUSS_DIR / "wsh_r248x576_gaussianized_core.npy")  # (2192, 248, 576)
dt = 3.0 * 3600.0
T = core.shape[0]
print(f"core shape {core.shape}")


## Sweep `|lambda_k|` across DMD rank and across sliding sub-windows

In [ ]:
import pandas as pd

from train.dmd import dmd_fit

print("--- across r_dmd, full window (reusing task 07's rank-sweep CSV, not recomputed) ---")
sweep = pd.read_csv(GAUSS_DIR / "dmd_rank_sweep_wsh_r124x288_202401-202410.csv")
sweep = sweep[sweep["r_dmd"].isin([5, 10, 20, 40])]
print(sweep[["r_dmd", "n_growing", "max_lambda"]].to_string(index=False))
# NOTE: this is task 07's 20% Tucker core (r124x288), kept as originally
# computed -- not the 40% production core used below for step 3, since
# recomputing on the 40% core would just cost extra compute to check the
# same r_dmd values this CSV already covers.

print("\n--- r_dmd=100 (production rank), sliding sub-windows of the production 40% core ---")
r_dmd = 100
window_len = T // 3
for start in range(0, T - window_len, window_len // 2):
    sub_core = core[start : start + window_len]
    dmd = dmd_fit(sub_core, r_dmd=r_dmd, dt=dt, rng=rng)
    growing = np.abs(dmd.Lambda) > 1.0 + 1e-6
    print(f"window [{start}:{start + window_len}]: {growing.sum()} growing modes, "
          f"max|lambda|={np.abs(dmd.Lambda).max():.4f}")


## Decision-rule diagnostic

A mode that is growing consistently across sub-windows and ranks is a
stronger candidate for "genuine" growth (e.g. a real, sustained storm
intensification within the window) than one whose growth appears/disappears
with rank or window choice (noise-attributable). Use this to calibrate the
"small margin: clip" vs. "large margin: flag for review" thresholds
`dynamic_mode_decomposition.md` §6 proposes. Record in
`tasks/09-growing-dmd-modes.md` §4.

**Caveat (2026-09-29, redefined onto task 07's artifacts):** task 07's own
rank-sweep already found `n_growing=0` at every tested rank on the 20%
core. If the production-core sliding-window sweep above also finds
nothing growing, that's an honest, uninformative null result, not a bug —
there may simply be no growing mode anywhere in this dataset to calibrate
the clip/flag threshold against. Per an explicit choice when this was
redefined, no era-3 fallback window is kept in this notebook; a genuine
growing-mode example to calibrate against would be a deliberate new
follow-on, not something this notebook produces automatically.
